# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [57]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [58]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [59]:
# Task 1
# Heatmap: number of titles by rating and decade
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'
ratings_keep = ['TV-14','TV-MA','PG-13','R','PG']
df1 = df[df['rating'].isin(ratings_keep)].copy()
pivot = df1.groupby(['rating','decade']).size().unstack(fill_value=0).reindex(index=ratings_keep).fillna(0)
import plotly.express as px
fig = px.imshow(pivot.values,
                x=pivot.columns,
                y=pivot.index,
                color_continuous_scale='Blues',
                text_auto=True,
                labels=dict(x='Decade', y='Rating'))
fig.update_layout(title='Which ratings dominate each decade — TV-14 leads the 2010s and 2020s')
fig.show()

## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [60]:
# Task 2
# Waterfall: Movie additions per year (2015-2022)
df_movies = df[df['type'] == 'Movie'].copy()
# create 'added_year' if missing (from date_added or fallback to release_year)
if 'added_year' not in df_movies.columns:
    if 'date_added' in df_movies.columns:
        df_movies['added_year'] = pd.to_datetime(df_movies['date_added'], errors='coerce').dt.year
    else:
        df_movies['added_year'] = df_movies.get('release_year')
years = list(range(2015, 2023))
df_years = df_movies[df_movies['added_year'].between(2015, 2022)].copy()
counts = df_years.groupby('added_year').size().reindex(years, fill_value=0)
total = int(counts.sum())
x_labels = [str(y) for y in years] + ['Total']
y_values = list(map(int, counts.values)) + [total]
measures = ['relative'] * len(years) + ['total']
# show values on each bar (inside) and style text for readability
wf = go.Waterfall(
    name='Movie additions',
    orientation='v',
    measure=measures,
    x=x_labels,
    y=y_values,
    text=y_values,
    textposition='inside',
    textfont=dict(color='white', size=11),
    increasing=dict(marker=dict(color='green')),
    decreasing=dict(marker=dict(color='red')),
    totals=dict(marker=dict(color='blue'))
)
fig = go.Figure(wf)
# annotate the largest single-year addition and adjust styling
if counts.sum() > 0:
    max_year = int(counts.idxmax())
    max_val = int(counts.max())
    # place annotation near the top of the bar (inside) with contrasting style
    #fig.add_annotation(x=str(max_year), y=max_val * 0.9, text=f'Max addition: {max_val} titles ({max_year})', showarrow=False, font=dict(color='white'), bgcolor='rgba(0,0,0,0.6)', yanchor='top')
else:
    max_year = None
    max_val = 0
fig.update_layout(title_text=f'Netflix movie library growth (2015–2022) — peak in {max_year if max_year else 'N/A'}', xaxis_title='Year', yaxis_title='Number of titles', template='plotly_white', xaxis=dict(tickmode='array', tickvals=x_labels, tickangle=0))
fig.show()
# Dynamic insight printed below the chart
from IPython.display import Markdown, display
display(Markdown(f'**Insight:** Netflix added {total} movies between 2015–2022, with the largest single-year addition in {max_year if max_year else 'N/A'} ({max_val} titles).'))

**Insight:** Netflix added 659 movies between 2015–2022, with the largest single-year addition in 2016 (93 titles).